# 3 · Audio segmentation & forced alignment (MMS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/03_segmentation/03_mms_segmentation.ipynb)

**Pipeline stage 3 of 6.** The recordings are 2–5-minute narratives with no
internal time alignment, whereas CTC training needs short, aligned utterances.
We use **Meta's MMS forced aligner** to get word-level timestamps, then compare
three segmentation strategies:

1. **10-word** fixed windows — long, context-rich, but cut across pauses.
2. **5-word** fixed windows — short, uniform, tighter alignment.
3. **Prosodic (breath-group)** — cut at boundary signals in the orthography;
   **adopted** because boundaries coincide with real acoustic pauses.

## Setup

In [ ]:
!pip -q install torch torchaudio transformers soundfile librosa
import torch, torchaudio, soundfile as sf, os, glob, json, re
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device)

In [ ]:
# ---- Global data layout (batch-ready; example ships 2 files) ----
import glob, pandas as pd
DATA="data"; AUDIO=f"{DATA}/audio"; TRANS=f"{DATA}/transcripts"
SEG=f"{DATA}/segments"; INDEX=f"{DATA}/index.csv"
os.makedirs(SEG, exist_ok=True)
index = pd.read_csv(INDEX)

def read_body(path, drop_header=2):
    """Spoken text = transcript minus the header + title lines."""
    lines=[l for l in open(path,encoding="utf-8").read().splitlines() if l.strip()]
    return " ".join(lines[drop_header:])
print(len(index),"recording(s):", list(index["id"]))

## 1 · Standardise audio (mono, 16 kHz)

In [ ]:
TARGET_SR = 16000
def load_audio(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1: wav = wav.mean(axis=1)         # to mono
    wav = torch.tensor(wav, dtype=torch.float32)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav / (wav.abs().max() + 1e-9)            # peak normalise
    return wav

## 2 · Forced alignment with MMS

We use torchaudio's MMS forced-alignment bundle (`MMS_FA`, pre-trained on
1,100+ languages). It returns frame-level token alignments that we aggregate
into **word spans**. Romanisation handles the Latin-based Kölsch orthography.

In [ ]:
from torchaudio.pipelines import MMS_FA as bundle
aligner_model = bundle.get_model().to(device)
LABELS = bundle.get_labels()
DICT   = bundle.get_dict()

def align_words(wav, transcript):
    # Returns [(word, t_start_sec, t_end_sec), ...] using MMS forced alignment.
    words = transcript.lower().split()
    # map characters to token ids (chars not in dict are skipped)
    tokens = [[DICT[c] for c in w if c in DICT] for w in words]
    flat = [t for w in tokens for t in w]
    with torch.inference_mode():
        emission, _ = aligner_model(wav.unsqueeze(0).to(device))
    targets = torch.tensor([flat], dtype=torch.int32, device=device)
    aligned, scores = torchaudio.functional.forced_align(emission, targets, blank=0)
    spans = torchaudio.functional.merge_tokens(aligned[0], scores[0])
    ratio = wav.size(0) / emission.size(1) / TARGET_SR
    # walk spans back into per-word time ranges
    out, k = [], 0
    for w, toks in zip(words, tokens):
        if not toks:        # word had no alignable chars
            continue
        seg = [s for s in spans if s.token != 0][k:k+len(toks)]
        if not seg:
            continue
        out.append((w, seg[0].start*ratio, seg[-1].end*ratio)); k += len(toks)
    return out
# NOTE: forced_align APIs vary across torchaudio versions; see README for the
# `ctc-forced-aligner` alternative if your version differs.

## 3 · Three segmentation strategies

### (a) Fixed windows — 5-word and 10-word

In [ ]:
def fixed_windows(word_spans, n=5):
    chunks = []
    for i in range(0, len(word_spans), n):
        grp = word_spans[i:i+n]
        if grp:
            chunks.append((grp[0][1], grp[-1][2], " ".join(w for w,_,_ in grp)))
    return chunks   # [(start, end, text), ...]

### (b) Prosodic (breath-group) segmentation — adopted

Cut at boundary signals already present in the orthography:
- **Hard** boundaries (full breath reset) at sentence-final punctuation `. ! ? ;`
- **Soft** boundaries (clause pause) at a comma before a Kölsch clause-starter
  (`un, dann, da, do, wie, ävver, odder, weil, dat, wenn, als, so, su`) or after
  a closing discourse particle (`ne, jo, ja, also`).

Then merge chunks < 3 words and split chunks > 12 words (target band 3–10).

In [ ]:
CLAUSE_STARTERS = {"un","dann","da","do","wie","ävver","odder","weil","dat",
                   "wenn","als","so","su"}
CLOSERS = {"ne","jo","ja","also"}

def prosodic_chunks(word_spans, raw_text):
    # align punctuation cues from raw_text onto the word stream
    raw_words = raw_text.split()
    boundaries = set()
    for i, rw in enumerate(raw_words):
        clean = re.sub(r"[^\wäöüßçəɪɛɔʊʁ']", "", rw.lower())
        if re.search(r"[.!?;]$", rw):                       # hard
            boundaries.add(i)
        elif rw.endswith(",") and (i+1 < len(raw_words) and
             re.sub(r"\W","",raw_words[i+1].lower()) in CLAUSE_STARTERS):
            boundaries.add(i)                                # soft
        elif clean in CLOSERS:
            boundaries.add(i)
    # cut the (aligned) word stream at those indices
    chunks, cur = [], []
    for i, ws in enumerate(word_spans):
        cur.append(ws)
        if i in boundaries:
            chunks.append(cur); cur = []
    if cur: chunks.append(cur)
    # post-process: merge <3, split >12
    out = []
    for ch in chunks:
        if out and len(ch) < 3:
            out[-1].extend(ch)
        elif len(ch) > 12:
            for j in range(0, len(ch), 10): out.append(ch[j:j+10])
        else:
            out.append(ch)
    return [(c[0][1], c[-1][2], " ".join(w for w,_,_ in c)) for c in out if c]

## 4 · Export segments + manifest

Each chunk is written as a 16-kHz WAV plus a manifest row
(`audio_path, text, start, end`) for the training notebook.

In [ ]:
import json
def export_segments(wav, chunks, out_dir, stem):
    os.makedirs(out_dir, exist_ok=True)
    rows=[]
    for k,(s,e,text) in enumerate(chunks):
        clip = wav[int(s*TARGET_SR):int(e*TARGET_SR)]
        p = os.path.join(out_dir, f"{stem}_{k:03d}.wav")
        sf.write(p, clip.numpy(), TARGET_SR)
        rows.append({"id": stem, "audio_path": p, "text": text, "start": s, "end": e})
    return rows

# GLOBAL: forced-align + segment every recording in the registry -> manifest.csv
# (needs the MMS aligner weights; downloads on first run.)
manifest=[]
for r in index.itertuples(index=False):
    wav   = load_audio(os.path.join(DATA, r.audio))
    body  = read_body(os.path.join(DATA, r.transcript))
    spans = align_words(wav, body)                 # MMS forced alignment
    chunks = prosodic_chunks(spans, body)          # adopted breath-group strategy
    manifest += export_segments(wav, chunks, SEG, r.id)
    print(f"{r.id}: {len(chunks)} segments from {len(body.split())} words")

if manifest:
    pd.DataFrame(manifest).to_csv(os.path.join(SEG,"manifest.csv"), index=False)
    print("wrote", os.path.join(SEG,"manifest.csv"), "->", len(manifest), "utterances")

## Why prosodic wins

Fixed windows are deterministic but ignore where the speaker breathes. Prosodic
segments place boundaries at real acoustic pauses, giving cleaner alignment
targets and lower downstream error. Discard any chunk that fails a
duration/speech-rate sanity check before training.